# PG-CFM on Colab: best 모델(blend_band) 학습·재현과 NeuralGCM 비교

이 노트북은 `docs/GPU_QUICKSTART.md`의 명령을 Colab에 맞게 옮긴 것입니다. 두 종류의 런타임을 씁니다.

| 부분 | 런타임 (메뉴 *런타임 → 런타임 유형 변경*) | 하는 일 |
| --- | --- | --- |
| A | **GPU** (무료 T4 16 GB, Pro면 L4·A100) | 우리 모델 학습·생성·보정·채점, ClimODE 체크포인트 비교, 2020년 멤버 |
| B | **TPU** (v5e-1) | NeuralGCM 공개 체크포인트 실행 (JAX라 TPU에서 그대로 돕니다) |
| C | 아무거나 | NeuralGCM 짝 비교, 그림, 결과 정리 |

**중요한 제약**
- 무료 세션은 길어야 12시간이고 유휴·브라우저 닫힘에 끊기며 `/content` 디스크가 초기화됩니다. 그래서 자료·체크포인트·결과는 모두 Google Drive(`/content/drive/MyDrive/pgcfm`)에 둡니다.
- 긴 학습은 `MAX_HOURS`(예: 8)로 잘라 돌립니다. 예산이 차면 상태를 저장하고 끝나므로(종료 코드 75) **같은 셀을 다시 실행하면 이어서** 돕니다.
- 우리 모델은 PyTorch라 TPU에서는 돌리지 않습니다(torch_xla 미검증). TPU는 B부분(NeuralGCM)에만 씁니다.
- Drive 용량: 자료는 해마다 약 0.65 GB(ClimODE 연도 2006~2018 + 2020 = 약 9 GB), 앙상블 npz는 해·arm마다 228 MB입니다. 무료 Drive 15 GB면 ClimODE 연도 전체가 들어가지만 여유는 적습니다. 모자라면 `PROFILE=small`(2012~2016 + 2017~18, 약 4.5 GB)로 시작하세요.

## A. GPU 런타임: 우리 모델

### A1. Drive 마운트와 작업 폴더

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/pgcfm'
os.makedirs(WORK, exist_ok=True)
os.environ['WORK'] = WORK
print('작업 폴더', WORK)

### A2. 코드와 환경 (세션마다 다시 실행)

In [ ]:
%cd /content
!test -d physical_guide_flowmatching || git clone -q https://github.com/nayehyeon61-glitch/physical_guide_flowmatching.git
%cd /content/physical_guide_flowmatching
!git fetch -q origin claude/project-thread-43gbu8 && git checkout -q claude/project-thread-43gbu8 && git pull -q origin claude/project-thread-43gbu8
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!python -m pytest -q tests 2>&1 | tail -n 3

### A3. 스모크 (GPU·자료·사슬 확인, 약 15분)

자료는 스크립트가 빠진 해만 내려받습니다(`DATA`가 Drive라 다음 세션에도 남습니다). `CACHE`는 세션 디스크에 둡니다(Drive 위 memmap은 느립니다).

In [ ]:
!SMOKE=1 BLEND=1 DEVICE=auto DATA=$WORK/data OUT=$WORK/runs/smoke CACHE=/content/cache bash scripts/run_step13_best.sh 2>&1 | tail -n 30

### A4. 본 실행 (ClimODE 연도 2006~2015 학습, best 모델까지, ClimODE 체크포인트 비교 포함)

백그라운드로 띄우고 아래 셀로 로그를 봅니다. `MAX_HOURS=8`이면 8시간 뒤 상태를 저장하고 멈추니, 세션이 끊기거나 멈추면 **이 셀을 다시 실행**하세요(끝난 단계는 건너뛰고 학습은 체크포인트에서 이어집니다).
T4에서는 단계마다 수 시간이 걸려 여러 세션에 나뉩니다. 용량이 모자라면 `PROFILE=small`로 바꾸세요.

In [ ]:
import subprocess, os
env = dict(os.environ, BLEND='1', PROFILE='climode', DEVICE='auto', MAX_HOURS='8', CLIMODE='1', CLIMODE_ICS='180',
           DATA=f"{os.environ['WORK']}/data", OUT=f"{os.environ['WORK']}/runs/climode_blend", CACHE='/content/cache')
os.makedirs(env['OUT'], exist_ok=True)
log = open(f"{env['OUT']}/run.log", 'a')
p = subprocess.Popen(['bash', 'scripts/run_step13_best.sh'], env=env, stdout=log, stderr=subprocess.STDOUT)
print('pid', p.pid, '로그', env['OUT'] + '/run.log')

In [ ]:
!tail -n 20 $WORK/runs/climode_blend/run.log; echo; cat $WORK/runs/climode_blend/manifest.jsonl 2>/dev/null | tail -n 5

### A5. 결과 읽기 (`docs/GPU_QUICKSTART.md` 6절)

In [ ]:
import json, os
O = f"{os.environ['WORK']}/runs/climode_blend"
for name in ['score_blend.json', 'score_best.json']:
    f = f'{O}/{name}'
    if not os.path.exists(f):
        print(name, '아직 없음'); continue
    S = json.load(open(f)); iz = S['vars'].index('z')
    for arm, s in S['arms'].items():  # rmse: [lead 24..144][var]
        z = [s['rmse'][k][iz] for k in (0, 2, 5)]
        c = [s['crps'][k][iz] for k in (0, 2, 5)] if 'crps' in s else []
        print(f'{name} {arm:42s} z500 RMSE 24/72/144 h ' + ' / '.join(f'{v:.0f}' for v in z)
              + ('   CRPS ' + ' / '.join(f'{v:.4f}' for v in c) if c else ''))

### A6. 2020년 멤버 (NeuralGCM 비교용, 고정 가중치, 재학습 없음)

NeuralGCM의 학습 기간(1979~2017/2019)에 우리 시험 해 2017~18이 들어가서 NeuralGCM 비교만 시험 해를 2020으로 옮깁니다. A4가 끝난 뒤 실행하세요.

In [ ]:
%%bash
set -e
cd /content/physical_guide_flowmatching
O=$WORK/runs/climode_blend; N=$O/ngcm; K=$WORK/data; E=$WORK/data/era5_data; mkdir -p $N
python scripts/wb2_to_kai.py --out $K --all-hours --years 2020
for f in pinn_project.pt pinn_project.json seqfm_dct.pt seqregress.pt dyn_gfeat.pt dyn_gfeat.json stats \
         seqfm_dct_pinn_ens_2016.npz seqregress_pinn_ens_2016.npz; do [ -e $N/$f ] || ln -s ../$f $N/$f; done
C="--kai $K --era5 $E --out $N --pinn $N/pinn_project.pt --save-ens --resume-ens --train-years 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 --val-year 2016 --test-years 2020 --device cuda --val-every 0 --patience 0 --max-hours 0 --val-members 0 --cache /content/cache"
python scripts/run_step13_seqfm.py $C --basis dct
python scripts/run_step13_seqfm.py $C --model regress
python scripts/run_step16_predict.py --kai $K --era5 $E --run $N --tag dyn_gfeat --val-year 2016 --test-years 2020
python scripts/run_step13_guide.py --ens $N/seqfm_dct_pinn --era5 $E --out $N --core $N/dyn_gfeat --reg $N/seqregress_pinn \
    --reg-mbm --blend --val-year 2016 --test-years 2020 --save-arms regm_band blend_band
ls -la $N/*_ens_2020.npz

## B. TPU 런타임 (v5e-1): NeuralGCM 공개 체크포인트

런타임 유형을 **TPU**로 바꾼 뒤 A1, A2를 다시 실행하고 아래를 실행합니다. 체크포인트(`gs://neuralgcm/models/v1/`, CC BY-SA 4.0)는 스크립트가 내려받고,
입력은 공개 ARCO ERA5(0.25°, 37층)에서 IC마다 약 1 GB를 읽어 모델 격자로 재격자합니다. 2.8° 결정론 → 1.4° 결정론 → 1.4° 확률론 8멤버 순서로, 각각 2020년 ClimODE 규약 IC 180개 전부입니다.
IC마다 결과를 저장하므로 끊기면 `--ic-offset`으로 이어서 돌린 뒤 `scripts/merge_climode_pred.py`로 합칩니다.

In [ ]:
!pip install -q neuralgcm gcsfs
import jax; print(jax.devices())
# TPU가 안 보이면: !pip install -q -U "jax[tpu]" 후 런타임 재시작

In [ ]:
%%bash
cd /content/physical_guide_flowmatching
N=$WORK/runs/climode_blend/ngcm; E=$WORK/data/era5_data; mkdir -p $N
[ -e $E/geopotential_500 ] || python scripts/wb2_to_kai.py --out $WORK/data --all-hours --years 2020
python scripts/run_neuralgcm_eval.py --era5 $E --out $N/neuralgcm_2_8 --model v1/deterministic_2_8_deg --year 2020 2>&1 | grep -v Warning | tail -n 5
python scripts/run_neuralgcm_eval.py --era5 $E --out $N/neuralgcm_1_4 --model v1/deterministic_1_4_deg --year 2020 2>&1 | grep -v Warning | tail -n 5
python scripts/run_neuralgcm_eval.py --era5 $E --out $N/neuralgcm_ens --model v1/stochastic_1_4_deg --year 2020 --members 8 2>&1 | grep -v Warning | tail -n 5

## C. NeuralGCM 짝 비교와 그림 (A6과 B가 끝난 뒤, 아무 런타임)

변수는 두 모델이 공유하는 z500·t850뿐입니다(NeuralGCM에는 t2m·u10·v10이 없음). 조건 비대칭(자료 0.25°·37층·39년·TPU 대 5.625°·12층·4~10년·GPU)은 `docs/step23_neuralgcm.md`의 표를 그대로 씁니다.

In [ ]:
%%bash
cd /content/physical_guide_flowmatching
O=$WORK/runs/climode_blend; N=$O/ngcm; E=$WORK/data/era5_data
for R in neuralgcm_2_8 neuralgcm_1_4 neuralgcm_ens; do
  [ -e $N/$R/neuralgcm_pred.npz ] || continue
  python scripts/compare_step13_climode.py --era5 $E --climode $N/$R/neuralgcm_pred.npz --ref-name "NeuralGCM ($R, public checkpoint)" \
      --arms $N/seqregress_pinn $N/seqfm_dct_pinn_regm_band $N/seqfm_dct_pinn_blend_band --core $N/dyn_gfeat --out $N/${R}_paired.json
  python scripts/plot_step13.py --era5 $E --scores $O/score_best.json --years 2020 --figures none --paired $N/${R}_paired.json --out $N/fig_$R
done
ls $N/*_paired.json $N/fig_*/*.png

In [ ]:
import json, os, glob
N = f"{os.environ['WORK']}/runs/climode_blend/ngcm"
for f in sorted(glob.glob(f'{N}/*_paired.json')):
    d = json.load(open(f)); S = d['scores']
    print(os.path.basename(f), 'ICs', len(d.get('ics', [])), 'reference:', d.get('reference'))
    for arm, s in S.items():  # scores[arm][lead h][metric][var] = {mean, ci95}; "-climode" rows are paired differences
        row = []
        for var in ('z', 't'):
            m = [s[str(h)]['rmse'][var]['mean'] for h in (24, 72, 144)]
            row.append(var + ' ' + ' / '.join('nan' if v is None else f'{v:.3g}' for v in m))
        print(f'  {arm:50s} RMSE ' + '   '.join(row))

## 공유할 것

`$WORK/runs/climode_blend/{score_blend.json, score_best.json, energy_blend.json, climode_paired_best.json, manifest.jsonl, fig/*.png, ngcm/*_paired.json, ngcm/fig_*/*.png, ngcm/neuralgcm_*/neuralgcm_eval.json}`.
앙상블 npz는 공유하지 않아도 됩니다. 이 파일들을 올려 주시면 표와 그림을 문서에 채웁니다.